In [1]:
!nvidia-smi

Thu Jan  8 13:07:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      On  |   00000000:00:03.0 Off |                    0 |
| N/A   58C    P0             29W /   72W |    3952MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

In [3]:
from kernels.gla.triton.fwd_kernel_tf32 import parallel_gla_kernel

In [7]:
import torch
import triton


def triton_gla_forward(q, k, v, allow_tf32=True):
    q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
    B, L, H, D_QK = q.shape
    _, _, _, D_V = v.shape
    output = torch.empty_like(v)
    
    # Block Size for L4 / FP32
    # 32x32 float32 requires ~70KB, which fits comfortably in L4 (100KB limit).
    BLOCK_M = 32
    BLOCK_N = 32
    
    grid = (triton.cdiv(L, BLOCK_M), B * H)
    
    parallel_gla_kernel[grid](
        q, k, v, output,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        output.stride(0), output.stride(1), output.stride(2), output.stride(3),
        B, L, H,
        ALLOW_TF32=allow_tf32,
        BLOCK_M=BLOCK_M, 
        BLOCK_N=BLOCK_N, 
        D_QK=D_QK, 
        D_V=D_V,
        num_stages=2,
        num_warps=4
    )
    return output

In [8]:
B, L, H, D = 2, 1024, 4, 128
D_QK = 2 * D
dtype = torch.float32 
device = "cuda"

print(f"Benchmarking with: Batch={B}, Len={L}, Heads={H}, Dim={D}, Dtype={dtype}")

torch.manual_seed(0)
q_hat = torch.randn((B, L, H, D_QK), device=device, dtype=dtype)
k_hat = torch.randn((B, L, H, D_QK), device=device, dtype=dtype)
v     = torch.randn((B, L, H, D),    device=device, dtype=dtype)

print("\nVerifying Correctness (Strict FP32)...")

torch.backends.cuda.matmul.allow_tf32 = False

def pytorch_baseline(q, k, v):
    attn = torch.einsum('b l h d, b s h d -> b h l s', q, k)
    mask = torch.tril(torch.ones(L, L, device=q.device, dtype=torch.bool))
    attn = attn.masked_fill(~mask, 0.0)
    return torch.einsum('b h l s, b s h d -> b l h d', attn, v)

ref_out = pytorch_baseline(q_hat, k_hat, v)
tri_out = triton_gla_forward(q_hat, k_hat, v, allow_tf32=False)

diff = (ref_out - tri_out).abs()
max_diff = diff.max().item()
mean_val = ref_out.abs().mean().item()

print(f"   Output Mean Value: {mean_val:.4f}")
print(f"   Max Difference:    {max_diff:.4f}")

if max_diff < 1e-5: 
    print("PASS: Correctness verified.")
else:
    print(f"Diff is {100 * max_diff / mean_val:.2f}% of mean value.")

print("\nStarting Performance Benchmark (TF32 Enabled)...")

torch.backends.cuda.matmul.allow_tf32 = True

# Warmup
for _ in range(10):
    _ = triton_gla_forward(q_hat, k_hat, v, allow_tf32=True)
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

start_event.record()
n_loops = 100
for _ in range(n_loops):
    _ = triton_gla_forward(q_hat, k_hat, v, allow_tf32=True)
end_event.record()
torch.cuda.synchronize()

elapsed_time_ms = start_event.elapsed_time(end_event)
print(f"Average time per run: {elapsed_time_ms / n_loops:.4f} ms")

Benchmarking with: Batch=2, Len=1024, Heads=4, Dim=128, Dtype=torch.float32

Verifying Correctness (Strict FP32)...
   Output Mean Value: 271.5781
   Max Difference:    0.0000
PASS: Correctness verified.

Starting Performance Benchmark (TF32 Enabled)...
Average time per run: 0.2067 ms
